# 3D SOFFT Transformation Analysis

Interactive notebook for analyzing SO(3) registration results. Use the sliders to browse rotation candidates and volume slices.

In [1]:
# --- CONFIG ---
which_rotation = 0              # which rotation candidate (for all_solutions mode)
use_rotation_suffix = True      # True = 'Rotated0.csv' (all_solutions), False = 'Rotated.csv' (one_solution)

# --- DATA DIR ---
import os
DATA_DIR = 'data'
# If you move this notebook, update DATA_DIR to point to the folder with the CSV files.

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, IntSlider
import matplotlib
matplotlib.use('Agg')  # fallback only

def load_csv(filename):
    filepath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(filepath):
        print(f'File not found: {filepath}')
        return None
    return np.loadtxt(filepath)

print(f'Data directory: {DATA_DIR}')
print(f'Rotation candidate: {which_rotation}')

Data directory: data
Rotation candidate: 0


In [2]:
# --- Build rotated filenames from config ---
if use_rotation_suffix:
    rot_suffix = str(which_rotation)
else:
    rot_suffix = ''

# --- Load voxel data (N x N x N) ---
magnitude_fftw1 = load_csv('magnitudeFFTW1.csv')
phase_fftw1 = load_csv('phaseFFTW1.csv')
voxel_data_used1 = load_csv('voxelDataFFTW1.csv')

magnitude_fftw2 = load_csv('magnitudeFFTW2.csv')
phase_fftw2 = load_csv('phaseFFTW2.csv')
voxel_data_used2 = load_csv('voxelDataFFTW2.csv')

spectrum_real_fftw2_rotated = load_csv(f'spectrumRealFFTW2Rotated{rot_suffix}.csv')
spectrum_imag_fftw2_rotated = load_csv(f'spectrumImagFFTW2Rotated{rot_suffix}.csv')
voxel_data_used2_rotated = load_csv(f'voxelDataFFTW2Rotated{rot_suffix}.csv')

# --- Infer N and correlationN ---
if voxel_data_used1 is not None:
    N = int(round(len(voxel_data_used1) ** (1.0 / 3.0)))
    print(f'Inferred N = {N}')
else:
    raise ValueError('Cannot determine N: voxelDataFFTW1.csv not found')

magnitude_fftw2_rotated = np.sqrt(spectrum_real_fftw2_rotated ** 2 + spectrum_imag_fftw2_rotated ** 2)
phase_fftw2_rotated = np.arctan2(spectrum_imag_fftw2_rotated, spectrum_real_fftw2_rotated)
correlationN = int(round(len(magnitude_fftw2_rotated) ** (1.0 / 3.0)))
print(f'Inferred correlationN = {correlationN}  (expected N*2-1 = {N*2-1})')

# --- Reshape 3D data (C++ writes i,j,k order = row-major [i][j][k]) ---
magnitude1 = magnitude_fftw1.reshape((N, N, N))
phase1 = phase_fftw1.reshape((N, N, N))
voxel_data1 = voxel_data_used1.reshape((N, N, N))
magnitude2 = magnitude_fftw2.reshape((N, N, N))
phase2 = phase_fftw2.reshape((N, N, N))
voxel_data2 = voxel_data_used2.reshape((N, N, N))
magnitude2_rotated = magnitude_fftw2_rotated.reshape((correlationN, correlationN, correlationN))
phase2_rotated = phase_fftw2_rotated.reshape((correlationN, correlationN, correlationN))
voxel_data2_rotated = voxel_data_used2_rotated.reshape((N, N, N))

# --- Resampled magnitude (N x N) ---
resampled_magnitude1 = load_csv('resampledMagnitudeSO3_1.csv')
resampled_magnitude2 = load_csv('resampledMagnitudeSO3_2.csv')
resampled_magnitude1_matrix = resampled_magnitude1.reshape((N, N))
resampled_magnitude2_matrix = resampled_magnitude2.reshape((N, N))

print(f'All data loaded. Voxel shape: ({N},{N},{N}), Correlation shape: ({correlationN},{correlationN},{correlationN})')

Inferred N = 64
Inferred correlationN = 127  (expected N*2-1 = 127)
All data loaded. Voxel shape: (64,64,64), Correlation shape: (127,127,127)


## Input Voxel 3D Volume

3D volume rendering of the two input voxel scans using K3D with simple intensity thresholding.

In [3]:
# --- 3D Input Voxel Volume Rendering (K3D) ---
import k3d

# Plot 1: Voxel Data 1 (Reference)
plot1 = k3d.plot(camera_mode="orbit")
voxel1 = voxel_data1.astype(np.float32)
v1_max = voxel1.max()
threshold1 = v1_max * 0.05
volume1 = k3d.volume(
    voxel1,
    color_map=k3d.colormaps.basic_color_maps.Binary,
    alpha_coef=50,
    samples=500,
    color_range=[threshold1, v1_max]
)
plot1 += volume1
plot1.display()
print(f"Voxel 1: shape={voxel1.shape}, range=[{voxel1.min():.4f}, {v1_max:.4f}], threshold={threshold1:.4f}")

# Plot 2: Voxel Data 2 (Source)
plot2 = k3d.plot(camera_mode="orbit")
voxel2 = voxel_data2.astype(np.float32)
v2_max = voxel2.max()
threshold2 = v2_max * 0.05
volume2 = k3d.volume(
    voxel2,
    color_map=k3d.colormaps.basic_color_maps.Binary,
    alpha_coef=50,
    samples=500,
    color_range=[threshold2, v2_max]
)
plot2 += volume2
plot2.display()
print(f"Voxel 2: shape={voxel2.shape}, range=[{voxel2.min():.4f}, {v2_max:.4f}], threshold={threshold2:.4f}")


Output()

Voxel 1: shape=(64, 64, 64), range=[0.0000, 1.0000], threshold=0.0500


Output()

Voxel 2: shape=(64, 64, 64), range=[0.0000, 1.0000], threshold=0.0500


## Resampled Fourier Magnitudes

In [4]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Signal 1', 'Signal 2'))

fig.add_trace(
    go.Heatmap(z=resampled_magnitude1_matrix, colorscale='Viridis', showscale=False),
    row=1, col=1
)
fig.add_trace(
    go.Heatmap(z=resampled_magnitude2_matrix, colorscale='Viridis'),
    row=1, col=2
)

fig.update_layout(height=450, width=900, title_text='Resampled Magnitude Comparison')
fig.update_xaxes(nticks=20)
fig.update_yaxes(nticks=20)
fig.show()

## Sphere Projection of Magnitude

In [5]:
# --- Sphere Projection of Fourier Magnitude on S\u00b2 (K3D) ---
import k3d
import numpy as np

u = np.linspace(0, 2 * np.pi, N)
v = np.linspace(0, np.pi, N)

# Generate sphere vertices
vertices = np.zeros((N * N, 3))
for i in range(N):
    for j in range(N):
        idx = i * N + j
        vertices[idx, 0] = np.cos(u[i]) * np.sin(v[j])
        vertices[idx, 1] = np.sin(u[i]) * np.sin(v[j])
        vertices[idx, 2] = np.cos(v[j])

# Generate triangle indices (two triangles per quad)
indices = []
for i in range(N - 1):
    for j in range(N - 1):
        v0 = i * N + j
        v1 = (i + 1) * N + j
        v2 = (i + 1) * N + (j + 1)
        v3 = i * N + (j + 1)
        indices.append([v0, v2, v1])
        indices.append([v0, v3, v2])
indices = np.array(indices, dtype=np.uint32)

# Per-vertex attributes from resampled magnitude (transposed)
attr1 = resampled_magnitude1_matrix.T.flatten()
attr2 = resampled_magnitude2_matrix.T.flatten()

# Normalize to [0, 1] for color mapping
attr1_norm = attr1 / attr1.max()
attr2_norm = attr2 / attr2.max()

# Plot 1: Scan 1
plot1 = k3d.plot(name='Sphere 1', camera_mode='orbit')
mesh1 = k3d.mesh(
    vertices.astype(np.float32),
    indices,
    attribute=attr1_norm.astype(np.float32),
    color_map=k3d.colormaps.basic_color_maps.Jet,
    color_range=[0, 1],
    flat_shading=False,
)
plot1 += mesh1
plot1.display()
print(f"Sphere 1: {len(vertices)} vertices, magnitude range [{attr1.min():.4f}, {attr1.max():.4f}]")

# Plot 2: Scan 2
plot2 = k3d.plot(name='Sphere 2', camera_mode='orbit')
mesh2 = k3d.mesh(
    vertices.astype(np.float32),
    indices,
    attribute=attr2_norm.astype(np.float32),
    color_map=k3d.colormaps.basic_color_maps.Jet,
    color_range=[0, 1],
    flat_shading=False,
)
plot2 += mesh2
plot2.display()
print(f"Sphere 2: {len(vertices)} vertices, magnitude range [{attr2.min():.4f}, {attr2.max():.4f}]")


Output()

Sphere 1: 4096 vertices, magnitude range [2.2784, 15.8784]


Output()

Sphere 2: 4096 vertices, magnitude range [2.7412, 15.9608]


## SO(3) Correlation 3D Volume

Interactive 3D volume rendering with opacity based on correlation magnitude.

In [6]:
# --- Load SO(3) correlation volume (was in the removed slice-plot section) ---
correlation_values_real = load_csv('resultingCorrelationReal.csv')
correlation_values_complex = load_csv('resultingCorrelationComplex.csv')

corr_side = int(round(len(correlation_values_real) ** (1.0 / 3.0)))
bw_out = corr_side // 2
print(f'Correlation grid: {corr_side}x{corr_side}x{corr_side}  (bwOut = {bw_out})')

correlation_values_real_matrix = correlation_values_real.reshape((corr_side, corr_side, corr_side))
correlation_values_complex_matrix = correlation_values_complex.reshape((corr_side, corr_side, corr_side))
correlation_values_magnitude = np.sqrt(
    correlation_values_real_matrix ** 2 + correlation_values_complex_matrix ** 2
)

print(f'Correlation magnitude — max: {correlation_values_magnitude.max():.6f}, min: {correlation_values_magnitude.min():.6f}')

# --- SO(3) Correlation 3D Volume (K3D) ---
import k3d
import numpy as np

threshold = correlation_values_magnitude.mean() + correlation_values_magnitude.std()
vol_max = correlation_values_magnitude.max()
print(f"Volume threshold (mean+1std): {threshold:.4f}, max: {vol_max:.4f}")

# Normalize data for K3D
data_normalized = correlation_values_magnitude.astype(np.float32)

# Create the plot
plot = k3d.plot(camera_mode="orbit")

# Add volume rendering with JET colormap
# Low values are transparent, high values are bright/colored
volume = k3d.volume(
    data_normalized,
    color_map=k3d.colormaps.basic_color_maps.Jet,
    alpha_coef=50,
    samples=500,
    color_range=[threshold, vol_max]
)
plot += volume
plot.display()


Correlation grid: 64x64x64  (bwOut = 32)
Correlation magnitude — max: 898.087000, min: 842.843000
Volume threshold (mean+1std): 876.9122, max: 898.0870


Output()

## Translation Correlation Volume

In [7]:
# --- Translation Correlation Volume with Rotation Slider ---
import glob
import json

# Pre-compute list of available rotation candidates
shift_files = sorted(glob.glob(os.path.join(DATA_DIR, 'resultingCorrelationShift*.csv')))
file_rot_ids = sorted(int(os.path.basename(f).replace('resultingCorrelationShift', '').replace('.csv', ''))
                      for f in shift_files)

# Restrict to rotation candidates with debug metadata (quaternion, error, etc.).
# The 88 shift files in data/ include 69 extra candidates from a looser
# level_potential_rotation run; they have no useful metadata. The C++ writer
# uses the same loop index p for the shift file ID and the solution entry,
# so solutions[i].rotation_index == i and the first 19 file IDs are valid.
metadata_path_tcv = '/home/tim-external/ros_ws/src/fsregistration/plotting_results/3d/debug_metadata.json'
with open(metadata_path_tcv) as _f:
    _meta_tcv = json.load(_f)
metadata_rot_ids = sorted(sol['rotation_index'] for sol in _meta_tcv['solutions'])
valid_rot_ids = [r for r in file_rot_ids if r in set(metadata_rot_ids)]
num_candidates = len(valid_rot_ids)
max_rot_idx = max(valid_rot_ids) if valid_rot_ids else 0
corrN = N * 2 - 1  # same for all rotations
print(f'Found {num_candidates} rotation candidates, correlation grid {corrN}x{corrN}x{corrN}')

@interact(rotation=(0, max_rot_idx, 1), slice_idx=(0, corrN - 1, 1))
def show_translation_correlation(rotation=0, slice_idx=63):
    data = load_csv(f'resultingCorrelationShift{rotation}.csv')
    if data is None:
        return
    vol = data.reshape((corrN, corrN, corrN))

    # Peak detection
    c_max = np.max(vol)
    idx_max = np.argmax(vol)
    peak_i, peak_j, peak_k = np.unravel_index(idx_max, vol.shape)
    center = corrN // 2
    tx, ty, tz = peak_i - center, peak_j - center, peak_k - center

    slice_data = vol[slice_idx, :, :]

    fig = make_subplots(rows=1, cols=2,
        specs=[[{'type': 'heatmap'}, {'type': 'scene'}]],
        subplot_titles=(f'XY Slice [{slice_idx}, :, :]', f'XY Slice [{slice_idx}, :, :]'))

    fig.add_trace(go.Heatmap(z=slice_data, colorscale='Hot', showscale=False), row=1, col=1)
    fig.add_trace(go.Scatter(x=[peak_k], y=[peak_j], mode='markers',
        marker=dict(size=14, color='red', symbol='x', line_width=2), name='Peak'), row=1, col=1)
    fig.add_trace(go.Surface(z=slice_data, colorscale='Hot', showscale=True,
        colorbar_title='Correlation'), row=1, col=2)

    fig.update_layout(height=500, width=1100,
        title_text=f'Translation Correlation \u2014 Rot {rotation}, Slice {slice_idx}/{corrN-1} | '
                   f'Peak ({peak_i},{peak_j},{peak_k}) \u2192 Translation ({tx},{ty},{tz})')
    fig.show()


Found 19 rotation candidates, correlation grid 127x127x127


interactive(children=(IntSlider(value=0, description='rotation', max=18), IntSlider(value=63, description='sli…

## Compare Rotation Candidates

In [8]:
# Quick comparison of translation peak heights across rotation candidates
import glob
import json

shift_files = glob.glob(os.path.join(DATA_DIR, 'resultingCorrelationShift*.csv'))
shift_pairs = []
for fpath in shift_files:
    fname = os.path.basename(fpath)
    rot_id = int(fname.replace('resultingCorrelationShift', '').replace('.csv', ''))
    shift_pairs.append((rot_id, fpath))
shift_pairs.sort(key=lambda x: x[0])

metadata_path_cmp = '/home/tim-external/ros_ws/src/fsregistration/plotting_results/3d/debug_metadata.json'
with open(metadata_path_cmp) as f:
    meta_cmp = json.load(f)
trans_lvl_for_rot = {}
for sol in meta_cmp.get('solutions', []):
    ri = sol['rotation_index']
    trans_lvl_for_rot[ri] = sol['translations'][0]['level_potential'] if sol['translations'] else None

print(f'Found {len(shift_pairs)} rotation candidates:\n')
print(f'{"Rot":>4}  {"Grid":>6}  {"Max Correlation":>16}  {"Peak Index":>20}  {"Translation (from center)":>30}  {"TransLevelPot":>14}')
print('-' * 100)

for rot_id, fpath in shift_pairs:
    data = np.loadtxt(fpath)
    sz = int(round(len(data) ** (1.0 / 3.0)))
    vol = data.reshape((sz, sz, sz))
    mx = np.max(vol)
    idx = np.unravel_index(np.argmax(vol), vol.shape)
    ctr = sz // 2
    tx, ty, tz = idx[0] - ctr, idx[1] - ctr, idx[2] - ctr
    lvl = trans_lvl_for_rot.get(rot_id)
    lvl_str = f'{lvl:>14.3e}' if lvl is not None else f'{"—":>14}'
    print(f'{rot_id:>4}  {sz:>6}  {mx:>16.6f}  ({idx[0]:>3}, {idx[1]:>3}, {idx[2]:>3})  ({tx:>3}, {ty:>3}, {tz:>3})  {lvl_str}')

Found 88 rotation candidates:

 Rot    Grid   Max Correlation            Peak Index       Translation (from center)   TransLevelPot
----------------------------------------------------------------------------------------------------
   0     127          1.000000  ( 60,  58,  53)  ( -3,  -5, -10)       1.000e+00
   1     127          1.000000  ( 70,  76,  52)  (  7,  13, -11)       1.000e+00
   2     127          1.000000  ( 60,  76,  60)  ( -3,  13,  -3)       1.000e+00
   3     127          1.000000  ( 61,  73,  60)  ( -2,  10,  -3)       1.000e+00
   4     127          1.000000  ( 71,  58,  62)  (  8,  -5,  -1)       1.000e+00
   5     127          1.000000  ( 73,  73,  60)  ( 10,  10,  -3)       1.000e+00
   6     127          1.000000  ( 56,  61,  62)  ( -7,  -2,  -1)       1.000e+00
   7     127          1.000000  ( 69,  61,  61)  (  6,  -2,  -2)       1.000e+00
   8     127          1.000000  ( 68,  75,  54)  (  5,  12,  -9)       1.000e+00
   9     127          1.000000  ( 64, 

## Registered Point Cloud Overlay

Browse rotation candidates and their translation solutions. Use the two sliders to explore:

- **Rotation candidate**: which SO(3) rotation peak from the correlation
- **Translation candidate**: which translation solution within that rotation

- **Green**: Reference scan (Scan 1)
- **Blue**: Source scan (Scan 2) aligned by the selected rotation + translation

In [9]:
# --- Registered Point Cloud Overlay with Two Sliders ---
import json
import glob
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact

# Load metadata for GT and rotation info
metadata_path = '/home/tim-external/ros_ws/src/fsregistration/plotting_results/3d/debug_metadata.json'
with open(metadata_path) as f:
    meta = json.load(f)

voxel_size = meta['voxel_size']
solutions = meta['solutions']

gt_RPY = meta['gt_rpy_xyz']
gt_str = f"GT: RPY=({gt_RPY['roll_deg']:.1f}\u00b0, {gt_RPY['pitch_deg']:.1f}\u00b0, {gt_RPY['yaw_deg']:.1f}\u00b0) "
gt_str += f"T=({gt_RPY['x']:.3f}, {gt_RPY['y']:.3f}, {gt_RPY['z']:.3f})m"

# Build rotation order and pre-compute multiple translation peaks (top-N)
corrN = N * 2 - 1
shift_files = sorted(glob.glob(os.path.join(DATA_DIR, 'resultingCorrelationShift*.csv')))

rotation_order = []
translation_map = {}
max_trans = 0

def find_top_translation_peaks(rotation, n_peaks=10):
    data = load_csv(f'resultingCorrelationShift{rotation}.csv')
    if data is None:
        return []
    vol = data.reshape((corrN, corrN, corrN))
    vol_max = vol.max()
    vol_work = vol.copy()
    peaks = []
    for _ in range(n_peaks):
        idx_max = np.argmax(vol_work)
        corr = vol_work.flat[idx_max]
        if corr <= 0:
            break
        pi, pj, pk = np.unravel_index(idx_max, vol_work.shape)
        center = corrN // 2
        tx = (pi - center) * voxel_size
        ty = (pj - center) * voxel_size
        tz = (pk - center) * voxel_size
        peaks.append({'tx': tx, 'ty': ty, 'tz': tz, 'corr': corr, 'persist': 1.0})
        # Suppress 5x5x5 neighborhood
        s = 2
        i0, i1 = max(0, pi-s), min(corrN, pi+s+1)
        j0, j1 = max(0, pj-s), min(corrN, pj+s+1)
        k0, k1 = max(0, pk-s), min(corrN, pk+s+1)
        vol_work[i0:i1, j0:j1, k0:k1] = 0
    return peaks

# rotation_order is driven by metadata, not by glob — only rotation peaks
# with full debug metadata (quaternion, error, translations) are exposed.
# The 88 shift files in data/ include 69 extra candidates from a looser
# level_potential_rotation run; they have no useful metadata.
metadata_rot_ids = {sol['rotation_index'] for sol in solutions}
rotation_order = []
for fpath in shift_files:
    rot_id = int(os.path.basename(fpath).replace('resultingCorrelationShift', '').replace('.csv', ''))
    if rot_id in metadata_rot_ids and rot_id not in rotation_order:
        rotation_order.append(rot_id)

rotation_order.sort()

for ridx in rotation_order:
    peaks = find_top_translation_peaks(ridx, n_peaks=10)
    peaks.sort(key=lambda x: x['corr'], reverse=True)
    translation_map[ridx] = peaks
    if len(peaks) > max_trans:
        max_trans = len(peaks)

max_trans = max(0, max_trans - 1)
max_rot = max(0, len(rotation_order) - 1)

print(f'Loaded {len(rotation_order)} rotation candidates, up to {max_trans + 1} translation peaks each')
print(f'Voxel size: {voxel_size:.6f} m')
print(gt_str)

# Threshold for point cloud extraction
threshold1 = voxel_data1.max() * 0.05
threshold2 = voxel_data2.max() * 0.05
threshold = max(threshold1, threshold2)
print(f'Point cloud threshold: {threshold:.4f}')

# Downsample limit
MAX_POINTS = 30000

# Cache for pre-rotated voxel data and transformed source points
prerotated_cache = {}
transformed_cache = {}

# --- Pre-compute GT-aligned source point cloud ---
# GT maps source(pcd1)->target(pcd2): p2 = gt_R @ p1 + gt_t
# To align pcd2 with pcd1, apply INVERSE: p1 = gt_R^T @ (p2 - gt_t)
# Voxel data is centered (mean-subtracted): p_c = p - mean
# Forward GT in centered coords: p2_c = gt_R @ p1_c + t_fwd_centered
#   where t_fwd_centered = gt_R @ mean1 + gt_t - mean2
# Inverse: p1_c = gt_R^T @ (p2_c - t_fwd_centered)
mean1 = np.array(meta['mean1'])
mean2 = np.array(meta['mean2'])
gt_transform = np.array(meta['gt_transform'])
gt_R = gt_transform[:3, :3]
gt_t = gt_transform[:3, 3]
t_fwd_centered = gt_R @ mean1 + gt_t - mean2

src2_raw = np.argwhere(voxel_data2 > threshold)
src2_meters = (src2_raw - N // 2) * voxel_size
gt_src_pts = (gt_R.T @ (src2_meters - t_fwd_centered).T).T

ref_raw = np.argwhere(voxel_data1 > threshold)
ref_pts = (ref_raw - N // 2) * voxel_size

# Downsample if needed
if len(ref_pts) > MAX_POINTS:
    idx = np.random.default_rng(seed=42).choice(len(ref_pts), MAX_POINTS, replace=False)
    ref_pts = ref_pts[idx]
if len(gt_src_pts) > MAX_POINTS:
    idx = np.random.default_rng(seed=42).choice(len(gt_src_pts), MAX_POINTS, replace=False)
    gt_src_pts = gt_src_pts[idx]

print(f"GT point cloud: {len(gt_src_pts)} source points, {len(ref_pts)} reference points")

@interact(rot_idx=(0, max_rot, 1), trans_idx=(0, max_trans, 1))
def show_registered_pointcloud(rot_idx=0, trans_idx=0):
    ridx = rotation_order[rot_idx]
    trans_list = translation_map[ridx]
    n_trans = len(trans_list)
    trans_idx = min(trans_idx, n_trans - 1)

    # Look up metadata for this rotation
    sol = next((s for s in solutions if s['rotation_index'] == ridx), None)
    if sol is not None:
        est_rpy = sol['estimated_rpy_xyz']
        err = sol['error']
        print(f"Rotation #{ridx}:")
        print(f"  {gt_str}")
        print(f"  Estimated: RPY=({est_rpy['roll_deg']:.1f}\u00b0, {est_rpy['pitch_deg']:.1f}\u00b0, {est_rpy['yaw_deg']:.1f}\u00b0) "
              f"T=({est_rpy['x']:.3f}, {est_rpy['y']:.3f}, {est_rpy['z']:.3f})m")
        print(f"  Error: RPE={err['rotation_error_deg']:.2f}\u00b0, TPE={err['translation_error_m']:.4f}m")
    else:
        print(f"Rotation #{ridx}: no metadata (only peaks 0-20 have debug metadata)")

    # Get selected translation offset
    trans = trans_list[trans_idx]
    t_offset = np.array([trans['tx'], trans['ty'], trans['tz']])
    print(f"  Translation #{trans_idx}: offset=({trans['tx']:.3f}, {trans['ty']:.3f}, {trans['tz']:.3f})m, "
          f"corr={trans['corr']:.2e}")

    # Load pre-rotated voxel data and apply translation
    cache_key = (ridx, trans_idx)
    if cache_key in transformed_cache:
        src_pts = transformed_cache[cache_key]
    else:
        if ridx not in prerotated_cache:
            data = load_csv(f'voxelDataFFTW2Rotated{ridx}.csv')
            if data is None:
                print(f'  WARNING: voxelDataFFTW2Rotated{ridx}.csv not found')
                prerotated_cache[ridx] = voxel_data2
            else:
                prerotated_cache[ridx] = data.reshape((N, N, N))
        voxel2_rot = prerotated_cache[ridx]
        src_raw = np.argwhere(voxel2_rot > threshold)
        src_pts = (src_raw - N // 2) * voxel_size + t_offset
        transformed_cache[cache_key] = src_pts

    # Create stacked figure with GT and Estimated alignment
    fig = make_subplots(rows=2, cols=1, specs=[[{'type': 'scene'}], [{'type': 'scene'}]],
                        subplot_titles=('GT Alignment', 'Estimated Alignment'),
                        vertical_spacing=0.05)

    # Top: GT Alignment (static)
    fig.add_trace(go.Scatter3d(
        x=ref_pts[:, 0], y=ref_pts[:, 1], z=ref_pts[:, 2],
        mode='markers',
        marker=dict(size=2, color='green', opacity=0.5),
        name='Reference (Scan 1)',
        hoverinfo='name'
    ), row=1, col=1)

    fig.add_trace(go.Scatter3d(
        x=gt_src_pts[:, 0], y=gt_src_pts[:, 1], z=gt_src_pts[:, 2],
        mode='markers',
        marker=dict(size=2, color='red', opacity=0.5),
        name='Source (Scan 2, GT-aligned)',
        hoverinfo='name'
    ), row=1, col=1)

    # Bottom: Estimated Alignment (responds to sliders)
    fig.add_trace(go.Scatter3d(
        x=ref_pts[:, 0], y=ref_pts[:, 1], z=ref_pts[:, 2],
        mode='markers',
        marker=dict(size=2, color='green', opacity=0.5),
        name='Reference (Scan 1)',
        hoverinfo='name'
    ), row=2, col=1)

    fig.add_trace(go.Scatter3d(
        x=src_pts[:, 0], y=src_pts[:, 1], z=src_pts[:, 2],
        mode='markers',
        marker=dict(size=2, color='blue', opacity=0.5),
        name='Source (Scan 2, aligned)',
        hoverinfo='name'
    ), row=2, col=1)

    # Build title
    if sol is not None:
        title = (f"Rot #{ridx}: RPE={err['rotation_error_deg']:.1f}\u00b0, "
                 f"TPE={err['translation_error_m']:.3f}m | "
                 f"Trans #{trans_idx} ({n_trans}): ({t_offset[0]:.3f}, {t_offset[1]:.3f}, {t_offset[2]:.3f})m")
    else:
        title = (f"Rot #{ridx} (no metadata) | "
                 f"Trans #{trans_idx} ({n_trans}): ({t_offset[0]:.3f}, {t_offset[1]:.3f}, {t_offset[2]:.3f})m")

    fig.update_layout(
        scene=dict(
            xaxis_title='X (m)',
            yaxis_title='Y (m)',
            zaxis_title='Z (m)',
            aspectmode='cube'
        ),
        scene2=dict(
            xaxis_title='X (m)',
            yaxis_title='Y (m)',
            zaxis_title='Z (m)',
            aspectmode='cube'
        ),
        title=title,
        height=1200,
        width=900,
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
    )

    fig.show()


Loaded 19 rotation candidates, up to 10 translation peaks each
Voxel size: 0.078773 m
GT: RPY=(161.6°, 62.9°, 158.0°) T=(-3.055, -0.482, 2.607)m
Point cloud threshold: 0.0500
GT point cloud: 1702 source points, 906 reference points


interactive(children=(IntSlider(value=0, description='rot_idx', max=18), IntSlider(value=0, description='trans…

## Debug Metadata Analysis

The following visualizations load from `debug_metadata.json` (generated by `debug3dRegistration.py`).
This provides richer information than the CSV files alone:
- GT transform for error computation
- Rotation peaks with quaternion coordinates
- Translation peaks with persistence values
- Per-solution error metrics

In [10]:
# --- Load debug metadata ---
import json
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

metadata_file = 'debug_metadata.json'

with open(metadata_file) as f:
    meta = json.load(f)
print(f"Loaded {metadata_file}")
print(f"  N: {meta['N']}, voxel_size: {meta['voxel_size']:.6f}")
print(f"  Sample index: {meta['sample_index']}")
print(f"  Rotation peaks: {meta['num_rotation_peaks']}")
print(f"  Params: {meta['params']}")

# GT transform
T = np.array(meta['gt_transform'])
gt_R = T[:3, :3]
gt_t = T[:3, 3]
gt_rpy = meta['gt_rpy_xyz']
print(f"\nGT Transform:")
print(f"  RPY: [{gt_rpy['roll_deg']:.2f}, {gt_rpy['pitch_deg']:.2f}, {gt_rpy['yaw_deg']:.2f}] deg")
print(f"  T:   [{gt_rpy['x']:.4f}, {gt_rpy['y']:.4f}, {gt_rpy['z']:.4f}] m")


Loaded debug_metadata.json
  N: 64, voxel_size: 0.078773
  Sample index: 2
  Rotation peaks: 19
  Params: {'use_clahe': True, 'r_min': 8, 'r_max': 24, 'set_r_manual': False, 'level_potential_rotation': 0.001, 'level_potential_translation': 0.01, 'normalization': 0, 'noise': 'None'}

GT Transform:
  RPY: [161.59, 62.88, 158.03] deg
  T:   [-3.0550, -0.4819, 2.6075] m


### Error Analysis: RPE and TPE

For each solution, we compute:
- **RPE** (Rotation Prediction Error): angular distance between estimated and GT rotation
- **TPE** (Translation Prediction Error): Euclidean distance between estimated and GT translation

In [11]:
if meta is None:
    print("No metadata loaded.")
else:
    # Sort solutions by rotation error
    errs = [(s['rotation_index'], s['error']['rotation_error_deg'], s['error']['translation_error_m']) for s in meta['solutions']]
    errs.sort(key=lambda x: x[1])
    
    indices, rpe_vals, tpe_vals = zip(*errs) if errs else ([], [], [])
    
    fig = make_subplots(rows=1, cols=2,
        subplot_titles=('Rotation Error (RPE)', 'Translation Error (TPE)'),
        specs=[[{'type': 'bar'}, {'type': 'bar'}]])
    
    colors_rpe = ['green' if v < 5 else 'orange' if v < 20 else 'red' for v in rpe_vals]
    colors_tpe = ['green' if v < 0.1 else 'orange' if v < 0.5 else 'red' for v in tpe_vals]
    
    fig.add_trace(go.Bar(
        x=list(indices), y=list(rpe_vals),
        marker_color=colors_rpe,
        name='RPE',
        text=[f'{v:.1f}°' for v in rpe_vals],
        textposition='outside',
    ), row=1, col=1)
    
    fig.add_trace(go.Bar(
        x=list(indices), y=list(tpe_vals),
        marker_color=colors_tpe,
        name='TPE',
        text=[f'{v:.3f}m' for v in tpe_vals],
        textposition='outside',
    ), row=1, col=2)
    
    fig.update_layout(
        height=500, width=1100,
        title_text='Error Metrics per Solution (green < 5°/0.1m, orange < 20°/0.5m, red ≥ 20°/0.5m)',
        showlegend=False,
    )
    fig.update_xaxes(title_text='Solution #', row=1, col=1)
    fig.update_xaxes(title_text='Solution #', row=1, col=2)
    fig.update_yaxes(title_text='RPE (deg)', row=1, col=1)
    fig.update_yaxes(title_text='TPE (m)', row=1, col=2)
    fig.show()


### Per-Solution Transform Summary

Complete transform matrix and RPY angles for each solution.

In [12]:
if meta is None:
    print("No metadata loaded.")
else:
    import pandas as pd
    
    rows = []
    for sol in meta['solutions']:
        rpy = sol['estimated_rpy_xyz']
        rot = sol['rotation']
        trans_info = sol['translations'][0] if sol['translations'] else {}
        
        rows.append({
            'Sol#': sol['rotation_index'],
            'Roll(°)': rpy['roll_deg'],
            'Pitch(°)': rpy['pitch_deg'],
            'Yaw(°)': rpy['yaw_deg'],
            'X(m)': rpy['x'],
            'Y(m)': rpy['y'],
            'Z(m)': rpy['z'],
            'RotCorr': rot['correlation_height'],
            'Persist': rot['persistence'],
            'LevelPot': rot['level_potential'],
            'TransX(m)': trans_info.get('x_translation', 0),
            'TransY(m)': trans_info.get('y_translation', 0),
            'TransZ(m)': trans_info.get('z_translation', 0),
            'TransCorr': trans_info.get('correlation_height', 0),
            'GlobalTransCorr': trans_info.get('global_correlation_height', 0),
            'TransLevelPot': trans_info.get('level_potential', 0),
            'RPE(°)': sol['error']['rotation_error_deg'],
            'TPE(m)': sol['error']['translation_error_m']
        })
    
    df = pd.DataFrame(rows)
    
    def color_error(val):
        if pd.isna(val):
            return ''
        if val < 1.0:
            return 'color: green; font-weight: bold'
        elif val < 5.0:
            return 'color: orange'
        else:
            return 'color: red'
    
    styled = df.style.map(color_error, subset=['RPE(°)', 'TPE(m)']).format({'TransCorr': '{:.3e}', 'GlobalTransCorr': '{:.3e}', 'TransLevelPot': '{:.3e}'})
    
    print(f"\nPer-solution transforms (N={meta['N']}, voxel_size={meta['voxel_size']:.6f})")
    print(f"Total time: {meta['total_time_seconds']:.2f}s\n")
    display(styled)



Per-solution transforms (N=64, voxel_size=0.078773)
Total time: 196.37s



,Sol#,Roll(°),Pitch(°),Yaw(°),X(m),Y(m),Z(m),RotCorr,Persist,LevelPot,TransX(m),TransY(m),TransZ(m),TransCorr,GlobalTransCorr,TransLevelPot,RPE(°),TPE(m)
0,0,0.000000,0.000000,-90.000000,0.229071,-0.122866,0.726195,1.000000,inf,1.000000,0.081135,0.385643,0.704650,2.498e+17,9.888e-01,1.000e+00,138.973794,3.801746
1,1,1.637671,-5.382058,89.923021,1.211695,-0.220839,0.812930,0.984235,0.404119,0.158204,-0.549155,-1.011539,0.859559,1.726e+17,6.831e-01,1.000e+00,134.764612,4.636084
2,2,177.187500,0.000000,-95.625000,1.355090,-0.408854,4.823413,0.935799,0.351984,0.108495,0.236577,-1.033484,0.232578,1.546e+17,6.119e-01,1.000e+00,111.427944,4.936060
3,3,-2.812500,0.000000,-180.000000,-0.217122,1.169688,0.313812,0.906044,0.325839,0.087158,0.159491,-0.792412,0.240647,1.983e+17,7.850e-01,1.000e+00,178.260187,4.005272
4,4,-175.841198,-13.445335,83.884573,0.285574,-1.050748,4.836516,0.894188,0.313246,0.078456,-0.628679,0.380357,0.074259,1.850e+17,7.322e-01,1.000e+00,111.069800,4.056072
5,5,-174.615755,-1.630450,-0.076671,-0.702410,0.832451,4.950548,0.901747,0.292997,0.069807,-0.785788,-0.788085,0.231297,1.864e+17,7.378e-01,1.000e+00,173.802839,3.571033
6,6,174.482409,1.095688,179.947200,-0.655436,-0.228076,5.007191,0.880462,0.250214,0.048534,0.550813,0.148837,0.082660,1.773e+17,7.019e-01,1.000e+00,63.907611,3.403093
7,7,5.517591,-1.095688,-0.052800,-0.275172,0.230000,0.175243,0.849553,0.202339,0.029549,-0.471441,0.139927,0.155256,1.284e+17,5.082e-01,1.000e+00,116.458796,3.761650
8,8,-180.000000,-84.375000,-84.375000,1.012978,2.031811,2.464440,0.689039,0.099688,0.004718,-0.392380,-0.952060,0.703016,2.335e+17,9.243e-01,1.000e+00,166.373362,4.784114
9,9,-90.000000,-84.375000,-90.000000,-3.048768,1.056997,2.697761,0.685434,0.086451,0.003511,-0.074851,-0.799469,-0.546186,1.662e+17,6.578e-01,1.000e+00,176.198894,1.541557


### Per-Translation Peak Detail (Single Rotation)

Use the slider to pick a rotation; the table lists all translation peaks for that rotation, sorted by correlation height (highest first), with RPE and TPE per candidate.

In [ ]:
if meta is None:
    print("No metadata loaded.")
else:
    import pandas as pd
    from scipy.spatial.transform import Rotation as R_scipy
    n_rot = len(meta['solutions'])
    gt = np.asarray(meta['gt_transform'])
    mean1 = np.array(meta['mean1'])
    mean2 = np.array(meta['mean2'])
    
    def _compute_errors(gt, est):
        diff = np.linalg.inv(gt) @ est
        tpe = float(np.linalg.norm(diff[:3, 3]))
        r_mat = diff[:3, :3]
        r = R_scipy.from_matrix(r_mat)
        angle = r.as_rotvec()
        rpe = float(np.degrees(np.linalg.norm(angle)))
        return rpe, tpe
    
    @interact(rot_idx=(0, n_rot - 1, 1))
    def show_translation_peaks(rot_idx=0):
        sol = meta['solutions'][rot_idx]
        translations = sol.get('translations', [])
        if not translations:
            print(f"Rotation {rot_idx}: no translation peaks")
            return
        
        q = sol['rotation']
        rot_matrix = R_scipy.from_quat([q['x'], q['y'], q['z'], q['w']]).as_matrix()
        rows = []
        for t in translations:
            t_vec = np.array([t['x_translation'], t['y_translation'], t['z_translation']])
            est = np.eye(4)
            est[:3, :3] = rot_matrix
            est[:3, 3] = rot_matrix @ (t_vec - mean1) + mean2
            rpe, tpe = _compute_errors(gt, est)
            rows.append({
                'Peak#': t['index'],
                'X(m)': t['x_translation'],
                'Y(m)': t['y_translation'],
                'Z(m)': t['z_translation'],
                'TransCorr': t['correlation_height'],
                'GlobalTransCorr': t.get('global_correlation_height', 0),
                'Persist': t['persistence'],
                'TransLevelPot': t.get('level_potential', 0),
                'RPE(°)': rpe,
                'TPE(m)': tpe,
            })
        
        df = pd.DataFrame(rows).sort_values('TransCorr', ascending=False).reset_index(drop=True)
        
        def color_err(val):
            if pd.isna(val): return ''
            if val < 1.0: return 'color: green; font-weight: bold'
            if val < 5.0: return 'color: orange'
            return 'color: red'
        
        def color_tpe(val):
            if pd.isna(val): return ''
            if val < 0.1: return 'color: green; font-weight: bold'
            if val < 0.5: return 'color: orange'
            return 'color: red'
        
        styled = (df.style
                    .map(color_err, subset=['RPE(°)'])
                    .map(color_tpe, subset=['TPE(m)'])
                    .format({'TransCorr': '{:.3e}', 'GlobalTransCorr': '{:.3e}', 'TransLevelPot': '{:.3e}'}))
        print(f"Rotation {rot_idx}: {len(translations)} translation peak(s), sorted by correlation")
        display(styled)


interactive(children=(IntSlider(value=0, description='rot_idx', max=18), Output()), _dom_classes=('widget-inte…

## Overall Best Solution

The single candidate (rotation + translation pair) across **all** rotation
and translation peaks whose composed 4×4 transform is closest to the ground
truth. This uses the same combined metric as `testingSoftOnPredatorData.py`
`‖t‖ · angle`, which is the oracle-style "best" peak consumed by
`read_in_data_files.py`.

The "Highest Correlation" row shows the translation-correlation pick (max
`global_correlation_height` over all (rotation, translation) pairs), for comparison.

In [14]:
if meta is None:
    print("No metadata loaded.")
else:
    import pandas as pd
    from scipy.spatial.transform import Rotation as R_scipy
    
    gt = np.asarray(meta['gt_transform'])
    mean1 = np.array(meta['mean1'])
    mean2 = np.array(meta['mean2'])
    
    def _compute_errors(gt, est):
        diff = np.linalg.inv(gt) @ est
        tpe = float(np.linalg.norm(diff[:3, 3]))
        r_mat = diff[:3, :3]
        r = R_scipy.from_matrix(r_mat)
        angle = r.as_rotvec()
        rpe = float(np.degrees(np.linalg.norm(angle)))
        return rpe, tpe
    
    def _compute_combined_score(rpe, tpe):
        return tpe * np.radians(rpe)
    
    best = None
    for sol in meta['solutions']:
        q_rot = sol['rotation']
        rot_mat = R_scipy.from_quat([q_rot['x'], q_rot['y'], q_rot['z'], q_rot['w']]).as_matrix()
        for t in sol.get('translations', []):
            t_vec = np.array([t['x_translation'], t['y_translation'], t['z_translation']])
            est = np.eye(4)
            est[:3, :3] = rot_mat
            est[:3, 3] = rot_mat @ (t_vec - mean1) + mean2
            rpe, tpe = _compute_errors(gt, est)
            score = _compute_combined_score(rpe, tpe)
            if best is None or score < best[0]:
                best = (score, rpe, tpe, sol['rotation_index'], t['index'], est, t, sol)
    
    highest = None
    for sol in meta['solutions']:
        q_rot = sol['rotation']
        rot_mat = R_scipy.from_quat([q_rot['x'], q_rot['y'], q_rot['z'], q_rot['w']]).as_matrix()
        for t in sol.get('translations', []):
            t_vec = np.array([t['x_translation'], t['y_translation'], t['z_translation']])
            est = np.eye(4)
            est[:3, :3] = rot_mat
            est[:3, 3] = rot_mat @ (t_vec - mean1) + mean2
            rpe, tpe = _compute_errors(gt, est)
            gcorr = t.get('global_correlation_height')
            if gcorr is None:
                gcorr = t.get('correlation_height', 0)
            if highest is None or gcorr > highest[0]:
                highest = (gcorr, rpe, tpe, sol['rotation_index'], t['index'], est, t, sol)
    
    def color_err(val):
        if pd.isna(val): return ''
        if val < 1.0: return 'color: green; font-weight: bold'
        if val < 5.0: return 'color: orange'
        return 'color: red'
    
    def color_tpe(val):
        if pd.isna(val): return ''
        if val < 0.1: return 'color: green; font-weight: bold'
        if val < 0.5: return 'color: orange'
        return 'color: red'
    
    gt_rpy = meta['gt_rpy_xyz']
    
    rows = []
    
    if highest:
        corr, rpe, tpe, ri, ti, est, t, sol = highest
        rpy = sol['estimated_rpy_xyz']
        rows.append({
            'Sol#': f'{ri} (highest trans-corr)',
            'Peak#': ti,
            'Roll(°)': rpy['roll_deg'],
            'Pitch(°)': rpy['pitch_deg'],
            'Yaw(°)': rpy['yaw_deg'],
            'X(m)': est[0, 3],
            'Y(m)': est[1, 3],
            'Z(m)': est[2, 3],
            'RotCorr': sol['rotation']['correlation_height'],
            'TransCorr': t['correlation_height'],
            'GlobalTransCorr': t.get('global_correlation_height', 0),
            'Persist': t['persistence'],
            'TransLevelPot': t.get('level_potential', 0),
            'RPE(°)': rpe,
            'TPE(m)': tpe,
        })
    
    if best:
        score, rpe, tpe, ri, ti, est, t, sol = best
        rpy = sol['estimated_rpy_xyz']
        rows.append({
            'Sol#': f'{ri} (best)',
            'Peak#': ti,
            'Roll(°)': rpy['roll_deg'],
            'Pitch(°)': rpy['pitch_deg'],
            'Yaw(°)': rpy['yaw_deg'],
            'X(m)': est[0, 3],
            'Y(m)': est[1, 3],
            'Z(m)': est[2, 3],
            'RotCorr': sol['rotation']['correlation_height'],
            'TransCorr': t['correlation_height'],
            'GlobalTransCorr': t.get('global_correlation_height', 0),
            'Persist': t['persistence'],
            'TransLevelPot': t.get('level_potential', 0),
            'RPE(°)': rpe,
            'TPE(m)': tpe,
        })
    
    rows.append({
        'Sol#': 'GT',
        'Peak#': '—',
        'Roll(°)': gt_rpy['roll_deg'],
        'Pitch(°)': gt_rpy['pitch_deg'],
        'Yaw(°)': gt_rpy['yaw_deg'],
        'X(m)': gt_rpy['x'],
        'Y(m)': gt_rpy['y'],
        'Z(m)': gt_rpy['z'],
        'RotCorr': None,
        'TransCorr': None,
        'GlobalTransCorr': None,
        'Persist': None,
        'TransLevelPot': None,
        'RPE(°)': 0.0,
        'TPE(m)': 0.0,
    })
    
    df = pd.DataFrame(rows)
    
    styled = (df.style
                .map(color_err, subset=['RPE(°)'])
                .map(color_tpe, subset=['TPE(m)'])
                .format({'TransCorr': '{:.3e}', 'GlobalTransCorr': '{:.3e}', 'TransLevelPot': '{:.3e}', 'RotCorr': '{:.3e}'}))
    
    if best:
        print(f"Best candidate: rotation #{best[3]}, translation peak #{best[4]}")
        print(f"  Combined score = {best[0]:.6f} (TPE={best[2]:.4f}m, RPE={best[1]:.2f}°)")
    if highest:
        print(f"Highest translation correlation: rotation #{highest[3]}, translation peak #{highest[4]}")
        print(f"  GlobalTransCorr = {highest[0]:.3e}, TPE={highest[2]:.4f}m, RPE={highest[1]:.2f}°")
    print()
    display(styled)


Best candidate: rotation #17, translation peak #1
  Combined score = 0.006827 (TPE=0.1244m, RPE=3.14°)
Highest translation correlation: rotation #12, translation peak #0
  GlobalTransCorr = 1.000e+00, TPE=4.1705m, RPE=88.22°



,Sol#,Peak#,Roll(°),Pitch(°),Yaw(°),X(m),Y(m),Z(m),RotCorr,TransCorr,GlobalTransCorr,Persist,TransLevelPot,RPE(°),TPE(m)
0,12 (highest trans-corr),0,180.000000,84.375000,90.000000,0.158798,2.164569,2.360642,6.739e-01,2.526e+17,1.000e+00,1.000000,1.000e+00,88.219122,4.170528
1,17 (best),1,159.948508,59.879896,157.122015,-2.969006,-0.435737,2.684597,6.798e-01,1.496e+17,5.924e-01,0.732102,1.650e-01,3.144172,0.124408
2,GT,—,161.588120,62.877308,158.026868,-3.055014,-0.481900,2.607467,nan,nan,nan,nan,nan,0.000000,0.000000
